In [8]:
import numpy as np
import random
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPClassifier
from IPython.display import display, clear_output
import ipywidgets as widgets

print("Libraries imported successfully!")

Libraries imported successfully!


In [9]:
WINNING_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

def check_winner(board):
    for a, b, c in WINNING_LINES:
        if board[a] != 0 and board[a] == board[b] == board[c]:
            return board[a]

    return 0

def is_draw(board):
    return all(cell != 0 for cell in board) and check_winner(board) == 0

print("Game rules created successfully!")

Game rules created successfully!


In [10]:
def minimax(board, player):
    result = check_winner(board)

    if result == -1:
        return 1
    if result == 1:
        return -1
    if all(cell != 0 for cell in board):
        return 0

    scores = []

    for i in range(9):
        if board[i] == 0:
            board[i] = player
            scores.append(minimax(board, -player))
            board[i] = 0

    return max(scores) if player == -1 else min(scores)


training_boards = []
training_moves = []
visited = set()

def generate_training_data(board, player):
    state = (tuple(board), player)

    if state in visited:
        return

    visited.add(state)

    if check_winner(board) != 0 or all(v != 0 for v in board):
        return

    if player == -1:
        best_score = -float("inf")
        best_moves = []

        for i in range(9):
            if board[i] == 0:
                board[i] = -1
                score = minimax(board, 1)
                board[i] = 0

                if score > best_score:
                    best_score = score
                    best_moves = [i]
                elif score == best_score:
                    best_moves.append(i)

        for move in best_moves:
            training_boards.append(board.copy())
            training_moves.append(move)

    for i in range(9):
        if board[i] == 0:
            board[i] = player
            generate_training_data(board, -player)
            board[i] = 0


generate_training_data([0] * 9, 1)

X_train = np.array(training_boards)
y_train = np.array(training_moves)

print("Training examples:", len(X_train))
print("Input shape:", X_train.shape)
print("Training data generated successfully!")

Training examples: 4144
Input shape: (4144, 9)
Training data generated successfully!


In [11]:
ann = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="lbfgs",
    max_iter=800,
    random_state=42
)

ann.fit(X_train, y_train)

print("ANN training completed successfully!")

ANN training completed successfully!


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [12]:
def ai_move(board):
    probabilities = ann.predict_proba(
        np.array(board).reshape(1, -1)
    )[0]

    legal_moves = [i for i in range(9) if board[i] == 0]

    def move_score(move):
        matching_class = np.where(ann.classes_ == move)[0]

        if len(matching_class) == 0:
            return -1

        return probabilities[matching_class[0]]

    return max(legal_moves, key=move_score)

print("ANN move prediction function ready!")

ANN move prediction function ready!


In [13]:
board = [0] * 9
game_over = False
human_wins = 0
ai_wins = 0
draws = 0

buttons = [
    widgets.Button(
        description=" ",
        layout=widgets.Layout(width="85px", height="75px"),
        style={"font_weight": "bold"}
    )
    for _ in range(9)
]

status = widgets.HTML(
    value="<h3>Your turn: X</h3>"
)

scoreboard = widgets.HTML(
    value="<b>You: 0 | ANN: 0 | Draws: 0</b>"
)

new_game_button = widgets.Button(
    description="New Game",
    button_style="success"
)

game_grid = widgets.GridBox(
    children=buttons,
    layout=widgets.Layout(
        grid_template_columns="85px 85px 85px",
        grid_gap="5px"
    )
)

def update_board():
    symbols = {1: "X", -1: "O", 0: " "}

    for i, button in enumerate(buttons):
        button.description = symbols[board[i]]
        button.disabled = game_over or board[i] != 0

    scoreboard.value = (
        f"<b>You: {human_wins} | ANN: {ai_wins} | Draws: {draws}</b>"
    )

def finish_check():
    global game_over, human_wins, ai_wins, draws

    result = check_winner(board)

    if result == 1:
        game_over = True
        human_wins += 1
        status.value = "<h3 style='color:green'>You win!</h3>"

    elif result == -1:
        game_over = True
        ai_wins += 1
        status.value = "<h3 style='color:red'>ANN wins!</h3>"

    elif is_draw(board):
        game_over = True
        draws += 1
        status.value = "<h3>It's a draw!</h3>"

    update_board()
    return game_over

def player_move(index):
    global game_over

    if game_over or board[index] != 0:
        return

    board[index] = 1

    if finish_check():
        return

    status.value = "<h3>ANN is thinking...</h3>"

    move = ai_move(board)
    board[move] = -1

    if not finish_check():
        status.value = "<h3>Your turn: X</h3>"

    update_board()

def reset_game(_=None):
    global board, game_over

    board = [0] * 9
    game_over = False

    status.value = "<h3>Your turn: X</h3>"
    update_board()

for i, button in enumerate(buttons):
    button.on_click(lambda _, index=i: player_move(index))

new_game_button.on_click(reset_game)

update_board()

display(
    widgets.VBox([
        widgets.HTML("<h2>ANN Tic-Tac-Toe</h2>"),
        status,
        game_grid,
        new_game_button,
        scoreboard
    ])
)

In [14]:
test_board = [
    1, 0, 0,
    0, -1, 0,
    0, 0, 1
]

print("Test board:")
print(np.array(test_board).reshape(3, 3))

predicted_move = ai_move(test_board)

print("\nANN predicted cell:", predicted_move + 1)
print("Row:", predicted_move // 3 + 1)
print("Column:", predicted_move % 3 + 1)

Test board:
[[ 1  0  0]
 [ 0 -1  0]
 [ 0  0  1]]

ANN predicted cell: 2
Row: 1
Column: 2
